# Step 50 — Canonical catalogue, event composition and exports

The canonical ObsPy catalogue preserves seven composition classes. W = surviving waveform record, P = picks, H = hypocenter; these are archive-content classes, not physical earthquake types or quality grades.

| Class | Waveform | Picks | Hypocenter |
|---|---|---|---|
| W_P_H | yes | yes | yes |
| W_P | yes | yes | no |
| W_H | yes | no | yes |
| W_ONLY | yes | no | no |
| P_H | no | yes | yes |
| P_ONLY | no | yes | no |
| H_ONLY | no | no | yes |

`event_class:` and explicit presence flags are stored as Event comments and survive QuakeML. The CSV/JSON manifest provides easy filtering without parsing XML comments; it also reports actual pick counts, which can differ from declared spine presence if input linkage is incomplete. P_H fixes the earlier case where a pick-only container receiving a hypocenter remained incorrectly labeled P_ONLY.

Step 52 already writes `EVENT_CLASS`, `WAVEFORM_PRESENT`, `PICKS_PRESENT`, `HYPOCENTER_PRESENT` and `ORIGIN_STATUS` to SEISAN Nordic comment records. An anchor time needed for an unlocated S-file is not a real hypocenter: inspect its ORIGIN_STATUS/HYPOCENTER_PRESENT flags. Step 53 reads the class comments. Do not encode these classes in the earthquake-type field.

The existing all-record and waveform-related QuakeML/pickle outputs remain. These are subsets of the same catalogue snapshot, not independently assembled catalogues or different QuakeML schema versions. All outputs stay under the configured external output root. Legacy data are unchanged.

Preferred magnitudes follow step 43 (PINAALL first, then usable summary value; legacy zeros provisionally missing), independently of preferred hypocenter. Rerun updated step 43 before this stage. Content class is not proof that a match is scientifically correct.

**Hypocentre association (`association_policy`).** The default, `picks_then_window`, links each hypocentre one-to-one in two stages:
1. **Through the picks.** It links to the step-32 event whose first P pick is 0–10 s after the origin (`max_first_arrival_s`). At Pinatubo the first-arrival travel time is 1.7 s median and 4.5 s at the 99th percentile.
2. **By record window.** Remaining hypocentres link to the waveform record whose window contains the origin, from 10 s before the record start (`origin_lead_s`) to the record end. If several qualify, the record starting closest to 9.5 s before the origin wins.

Why not match on the record start: triggered records start about 11 s before the first arrival, so origins normally fall about 9.5 s *after* the start, and a ±10 s window centred on the start misses roughly 40% of them. Each linked event gets a `hypocenter_link:picks|window` comment. A hypocentre whose origin lies inside a record already linked to another hypocentre (e.g. two earthquakes in one file) stays H_ONLY, with an `inside_record_of_event:` comment. The earlier time-only policies remain available: `one_to_one` (greedy, smallest absolute time difference) and `nearest` (may reuse hypocentres; each event then gets its own copy of the origins, with IDs `origin/<id>/copy-for/<event>` and a `hypocenter_shared:` comment), both bounded by `origin_time_tol` around the record start. Preserve unresolved waveform/pick/origin multiplicity in later review.


## Implementation and execution guide

All catalogue assembly, class handling, manifests and optional QuakeML subset code is defined below. Antelope catalogue and waveform indexing exports are in step 54. No step-50 Python module or alternate wrapper is required. Shared `lib.shared` functions remain imported to keep matching behavior consistent across notebooks; FLOVOpy and ObsPy remain library dependencies.

Run configuration, then the definition cells, then the main run. Definition cells do not assemble catalogues or create databases. Optional export switches remain false by default. Antelope dependencies are isolated in step 54. Production output paths retain their existing names so steps 51/52 continue to work.

### Assembly and matching

Step-32 containers supply the waveform/pick spine. Step-43 hypocenter groups are matched by representative time within `origin_time_tol`; `one_to_one` is a greedy minimum-|time difference| assignment, not global optimization. `nearest` may reuse a hypocenter; its origins are then copied per event under unique resource IDs. Unmatched hypocenters become H_ONLY records. W/P/H labels describe retained archive content, not a claim that one container equals one earthquake. Inspect provenance and multi-event recordings before scientific interpretation.

The pick index supplies original scientific pick identities. Step-32 PI2L waveform-overlay targets are for review plots and do not silently replace original low-gain pick IDs here. Origin/magnitude values inherit the selected stage-43 policy. Magnitude provenance remains separate from hypocenter provenance; no magnitude type is guessed.

### Outputs and reuse

The main run replaces all-record and waveform-related QuakeML/pickle outputs plus the class manifest. Pickle is for trusted local Python interchange; QuakeML is the shareable scientific representation. Optional class subsets are filtered views with retained resource IDs, not independent catalogues. Counts from overlapping subsets must not be added. Step 54 handles native CSS catalogue metadata and optional MiniSEED wfdisc indexing; see that notebook for dependencies and validation status.

Run steps 43 → 50 → 51/52 → 53 when updating magnitudes or catalogue classes. Merely reopening this notebook does not update existing outputs.

## Configuration

Inspect all input/output paths and settings. This cell resolves configuration; definitions and run cells follow.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'lib' / 'clock.py').is_file()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from lib.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    waveform_event_index=P['WAVEFORM_PICK_EVENT_INDEX'],
    waveform_pick_map=P['PICK_MAP_CSV'],
    pick_index=P['MERGED_PICK_INDEX'],
    hypo_event_index=P['HYPO71_EVENT_INDEX'],
    hypo_origin_index=P['HYPO71_ORIGIN_INDEX'],
    origin_time_tol=float(P['ORIGIN_TIME_TOL_S']),  # only for the time-only policies
    out_prefix=P['CATALOG_PREFIX'],
    default_net='XB',
    association_policy='picks_then_window',  # Or 'one_to_one' / 'nearest' (origin within ±origin_time_tol of record start).
    max_first_arrival_s=10.0,   # picks stage: first P pick 0-10 s after the origin
    origin_lead_s=10.0,         # window stage: origin up to 10 s before the record start, or inside the record
)
display(SETTINGS)

## Core event and composition helpers

The next cell defines functions only. It does not execute the workflow or write outputs.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pickle

import math

import pandas as pd

from obspy import UTCDateTime

from obspy.core.event import (
    Catalog,
    Comment,
    Event,
    Origin,
    Magnitude,
    Pick,
    ResourceIdentifier,
    WaveformStreamID,
)

from flovopy.core.event_utils import parse_seed_id
from lib.shared import (safe_series_get, associate_one_to_one_by_time, associate_nearest_by_time,
                                   associate_hypocenters)

def stabilize_comment_ids(event):
    """Give comments IDs derived from parent ID, position and text, not obspy's random UUIDs,
    so a rerun with unchanged inputs yields identical QuakeML (step 52 fingerprints it)."""
    import hashlib
    for owner in [event, *event.origins, *event.magnitudes]:
        for i, comment in enumerate(owner.comments):
            digest = hashlib.sha256(f'{owner.resource_id}|{i}|{comment.text}'.encode()).hexdigest()
            comment.resource_id = ResourceIdentifier('smi:local/pinatubo/comment/' + digest)

def build_pick(row, default_net="XB"):
    """
    Build an ObsPy Pick from a pick-index row.
    Returns None if no valid pick_id is present.
    """
    pid = safe_series_get(row, "pick_id")
    if pid is None:
        return None

    seed_id = safe_series_get(row, "seed_id")
    net, sta, loc, chan = parse_seed_id(seed_id)

    wid = WaveformStreamID(
        network_code=net or default_net,
        station_code=sta,
        location_code=loc or "",
        channel_code=chan,
    )

    return Pick(
        time=UTCDateTime(row["pick_time"]),
        phase_hint=safe_series_get(row, "phase"),
        waveform_id=wid,
        resource_id=ResourceIdentifier(f"pick/{pid}"),
    )

def origin_resource_id(origin_id, event_key=None):
    """QuakeML ID of a step-43 origin. A hypocentre attached to more than one event (association_policy='nearest')
    gets one copy per event, each with its own ID, because a resource ID must be unique in the catalogue."""
    rid = f"origin/{origin_id}"
    return ResourceIdentifier(rid if event_key is None else f"{rid}/copy-for/{event_key}")

def attach_preferred_magnitude(event, hypo_row, event_key=None):
    """Export the independently selected magnitude; absent/legacy zero stays absent."""
    value = hypo_row.get('preferred_magnitude')
    if value is None or pd.isna(value):
        return
    value = float(value)
    if not math.isfinite(value) or (value == 0 and bool(hypo_row.get('zero_magnitude_is_missing', True))):
        return
    source = hypo_row.get('preferred_magnitude_source', 'unspecified')
    origin = hypo_row.get('preferred_magnitude_origin_id')
    magnitude = Magnitude(mag=value, resource_id=ResourceIdentifier(str(event.resource_id) + '/magnitude'),
                          comments=[Comment(text=f'magnitude_source:{source}'),
                                    Comment(text='legacy_zero_magnitudes_treated_as_missing')])
    if origin is not None and not pd.isna(origin):
        magnitude.origin_id = origin_resource_id(origin, event_key)
    event.magnitudes.append(magnitude)
    event.preferred_magnitude_id = magnitude.resource_id

def build_origin(row, event_key=None):
    """
    Build an ObsPy Origin from a hypo-origin row.
    Assumes depth_km is in km and converts to meters.
    """
    origin = Origin(
        time=UTCDateTime(row["origin_time"]),
        latitude=float(row["latitude"]),
        longitude=float(row["longitude"]),
        depth=float(row["depth_km"]) * 1000.0,
        resource_id=origin_resource_id(row['origin_id'], event_key),
    )
    # HYPO71 RMS residual and number of readings, when the source gives them (Nordic RMS column).
    rms = pd.to_numeric(row.get("rms"), errors="coerce")
    nass = pd.to_numeric(row.get("nass"), errors="coerce")
    if pd.notna(rms) or pd.notna(nass):
        from obspy.core.event import OriginQuality
        origin.quality = OriginQuality(standard_error=float(rms) if pd.notna(rms) else None,
                                       used_phase_count=int(nass) if pd.notna(nass) else None)
    # UTC run: nominal clock uncertainty and the clock rule (see 10_conversion/clock_regimes.csv).
    unc = pd.to_numeric(row.get("clock_uncertainty_s"), errors="coerce")
    if pd.notna(unc):
        from obspy.core.event import QuantityError
        origin.time_errors = QuantityError(uncertainty=float(unc))
    if isinstance(row.get("clock_regime"), str) and row.get("clock_regime"):
        origin.comments.append(Comment(text=f"clock_regime:{row['clock_regime']}; raw_time:{row.get('raw_origin_time')}"))
    return origin

def set_preferred_origin(event, hypo_row, event_key=None):
    """Mark step 43's preferred origin (by origin_id; else by time; else the first) as preferred."""
    if not event.origins:
        return
    chosen, how = None, 'step43_origin_id'
    wanted = hypo_row.get('preferred_origin_id') if hypo_row is not None else None
    if wanted is not None and not pd.isna(wanted):
        wanted_id = str(origin_resource_id(wanted, event_key))
        chosen = next((o for o in event.origins if str(o.resource_id) == wanted_id), None)
    if chosen is None and hypo_row is not None and not pd.isna(hypo_row.get('preferred_origin_time')):
        how = 'step43_origin_time'
        t = UTCDateTime(pd.Timestamp(hypo_row['preferred_origin_time']).isoformat())
        chosen = next((o for o in event.origins if abs(o.time - t) < 1e-3), None)
    if chosen is None:
        chosen, how = event.origins[0], 'first_origin_fallback'
    event.preferred_origin_id = chosen.resource_id
    source = hypo_row.get('preferred_origin_source') if hypo_row is not None else None
    event.comments.append(Comment(text=f"preferred_origin:{how}"
                                       + (f"; source:{source}" if isinstance(source, str) and source else '')))

EVENT_CLASS_FLAGS = {
    'W_P_H': (True, True, True), 'W_P': (True, True, False),
    'W_H': (True, False, True), 'W_ONLY': (True, False, False),
    'P_H': (False, True, True), 'P_ONLY': (False, True, False),
    'H_ONLY': (False, False, True),
}

def event_class_label(event):
    labels = [c.text.split(':', 1)[1] for c in event.comments if (c.text or '').startswith('event_class:')]
    if len(labels) != 1 or labels[0] not in EVENT_CLASS_FLAGS:
        raise ValueError(f'{event.resource_id}: missing/ambiguous event class')
    return labels[0]

## Portable manifest and QuakeML export

The next cell defines functions only. It does not execute the workflow or write outputs.

In [ ]:
def export_classified_catalog(catalog, outdir, *, write_subsets=True):
    """Write a portable CSV/JSON membership manifest and optional QuakeML subsets."""
    import json
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    rows = []
    for event in catalog:
        label = event_class_label(event)
        w, p, h = EVENT_CLASS_FLAGS[label]
        mag = event.preferred_magnitude() or (event.magnitudes[0] if event.magnitudes else None)
        rows.append(dict(event_resource_id=str(event.resource_id), event_class=label,
                         waveform_present=w, picks_present=p, hypocenter_present=h,
                         n_actual_picks=len(event.picks), n_origins=len(event.origins),
                         preferred_origin_id=str(event.preferred_origin_id) if event.preferred_origin_id else None,
                         magnitude=mag.mag if mag else None,
                         magnitude_comments=[c.text for c in mag.comments] if mag else [],
                         waveform_files=[c.text.split(':', 1)[1] for c in event.comments if (c.text or '').startswith('wavfile:')]))
    pd.DataFrame(rows, columns=['event_resource_id','event_class','waveform_present','picks_present',
        'hypocenter_present','n_actual_picks','n_origins','preferred_origin_id','magnitude',
        'magnitude_comments','waveform_files']).to_csv(outdir / 'event_manifest.csv', index=False)
    (outdir / 'event_manifest.json').write_text(json.dumps(rows, indent=2, allow_nan=False) + '\n')
    paths = []
    if write_subsets:
        subsets = {label: Catalog(events=[e for e in catalog if event_class_label(e) == label]) for label in EVENT_CLASS_FLAGS}
        subsets['located'] = Catalog(events=[e for e in catalog if EVENT_CLASS_FLAGS[event_class_label(e)][2]])
        for label, subset in subsets.items():
            if subset:
                path = outdir / ('catalog_' + label + '.xml')
                subset.write(str(path), format='QUAKEML')
                paths.append(str(path))
    return dict(manifest=rows, quakeml_paths=paths)

## Catalogue assembly

The next cell defines functions only. It does not execute the workflow or write outputs.

In [ ]:
def run(
    *,
    waveform_event_index,
    waveform_pick_map,
    pick_index,
    hypo_event_index,
    hypo_origin_index,
    origin_time_tol=10.0,
    out_prefix,
    default_net='XB',
    association_policy='picks_then_window',
    max_first_arrival_s=10.0,
    origin_lead_s=10.0,
):
    """Build both catalog subsets using one shared assembly/export path.

    association_policy='picks_then_window' (default) links each hypocentre
    through its picks (first P pick 0..max_first_arrival_s after the origin),
    then, for the rest, by record window (origin from origin_lead_s before
    the record start to its end); greedy one-to-one, see
    lib.shared.associate_hypocenters. Triggered records start
    ~11 s before the first arrival, so a window centred on the record start
    misses many origins.
    'one_to_one' uses deterministic greedy matching of origin time to record
    start within ±origin_time_tol; 'nearest' retains the original
    reuse-permitting nearest-time policy.
    This applies to the entire Step-32 spine, including pick-only events.
    Outputs keep the canonical Step-50 prefix consumed by steps 51/52.
    Return catalogs, association table, class counts and chosen policy.
    """
    if association_policy not in ('picks_then_window', 'one_to_one', 'nearest'):
        raise ValueError("association_policy must be 'picks_then_window', 'one_to_one' or 'nearest'")
    if not math.isfinite(origin_time_tol) or origin_time_tol < 0:
        raise ValueError("origin_time_tol must be finite and nonnegative")
    from types import SimpleNamespace
    args = SimpleNamespace(
        waveform_event_index=waveform_event_index,
        waveform_pick_map=waveform_pick_map,
        pick_index=pick_index,
        hypo_event_index=hypo_event_index,
        hypo_origin_index=hypo_origin_index,
        origin_time_tol=origin_time_tol,
        out_prefix=out_prefix,
        default_net=default_net,
    )

    # ------------------------------------------------------------------
    # Read inputs
    # ------------------------------------------------------------------

    df_evt = pd.read_csv(args.waveform_event_index, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})
    df_map = pd.read_csv(args.waveform_pick_map, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})
    df_pk = pd.read_csv(args.pick_index, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})
    df_he = pd.read_csv(args.hypo_event_index, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})
    df_ho = pd.read_csv(args.hypo_origin_index, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})

    # ------------------------------------------------------------------
    # Parse times
    # ------------------------------------------------------------------

    if "starttime" not in df_evt.columns:
        raise SystemExit("Step 50 requires 'starttime' column from Step 32")

    df_evt["event_time"] = pd.to_datetime(
        df_evt["starttime"], format="mixed", utc=True, errors="coerce"
    )
    if df_evt["event_time"].isna().any():
        raise SystemExit("Unparseable starttime values in Step 32 catalog")

    df_pk["pick_time"] = pd.to_datetime(
        df_pk["pick_time"], format="mixed", utc=True, errors="coerce"
    )
    if df_pk["pick_time"].isna().any():
        raise SystemExit("Unparseable pick_time values in pick index")

    df_he["preferred_origin_time"] = pd.to_datetime(
        df_he["preferred_origin_time"], format="mixed", utc=True, errors="coerce"
    )
    df_ho["origin_time"] = pd.to_datetime(
        df_ho["origin_time"], format="mixed", utc=True, errors="coerce"
    )

    # ------------------------------------------------------------------
    # Index helpers
    # ------------------------------------------------------------------

    picks_by_id = df_pk.set_index("pick_id", drop=False)
    map_by_event = df_map.groupby("event_id")

    # ------------------------------------------------------------------
    # Associate waveform events ↔ hypocenters
    # ------------------------------------------------------------------

    inside_record = {}
    if association_policy == 'picks_then_window':
        matches, inside_record = associate_hypocenters(
            df_evt, df_map, df_he, max_first_arrival_s=max_first_arrival_s, origin_lead_s=origin_lead_s,
        )
    elif association_policy == 'one_to_one':
        matches = associate_one_to_one_by_time(
            df_wave=df_evt, df_hypo=df_he,
            tolerance_seconds=args.origin_time_tol,
        )
    else:
        matches = associate_nearest_by_time(
            df_evt, df_he, tolerance_seconds=args.origin_time_tol,
        )

    hypo_for_waveform = dict(zip(matches["event_id_w"], matches["event_id_h"])) if not matches.empty else {}
    link_method = dict(zip(matches["event_id_w"], matches["method"])) if "method" in matches else {}
    used_hypo_ids = set(hypo_for_waveform.values())
    # Only 'nearest' can link one hypocentre to several events; those events get their own origin copies.
    events_for_hypo = {}
    for w, h in hypo_for_waveform.items():
        events_for_hypo.setdefault(h, []).append(w)
    shared_hypo = {h: ws for h, ws in events_for_hypo.items() if len(ws) > 1}
    if shared_hypo:
        print(f"Hypocenters linked to more than one event: {len(shared_hypo)} "
              f"({sum(map(len, shared_hypo.values()))} events); origin IDs are made unique per event")

    print(f"Hypocenters associated to waveform/pick events: {len(used_hypo_ids)}")
    if "method" in matches:
        print("  by method:", matches["method"].value_counts().to_dict())

    # ------------------------------------------------------------------
    # Build catalog_all
    # ------------------------------------------------------------------

    catalog_all = Catalog()
    class_counts = {
        "W_P_H": 0,
        "W_P": 0,
        "W_H": 0,
        "W_ONLY": 0,
        "P_ONLY": 0,
        "P_H": 0,
        "H_ONLY": 0,
    }

    # ---- Step 32 authoritative event spine ----
    for _, erow in df_evt.iterrows():
        eid = erow["event_id"]
        spine_class = safe_series_get(erow, "event_class", "")

        # Expected Step 32 classes:
        #   WAV_ONLY, WAV+PICKS, PICKS_ONLY
        has_w = spine_class in ("WAV_ONLY", "WAV+PICKS")
        has_p_spine = spine_class in ("PICKS_ONLY", "WAV+PICKS")

        ev = Event(resource_id=ResourceIdentifier(f"event/{eid}"))
        ev.comments = []

        # Provenance / metadata
        if has_w:
            ev.comments.append(
                Comment(text=f"waveform_starttime:{erow['event_time'].isoformat()}")
            )

        wav_file = safe_series_get(erow, "wav_file")
        if wav_file is None:
            wav_file = safe_series_get(erow, "waveform_file")

        if wav_file is not None:
            ev.comments.append(Comment(text=f"wavfile:{Path(str(wav_file)).name}"))

        # Picks
        if eid in map_by_event.groups:
            for _, prow in map_by_event.get_group(eid).iterrows():
                pid = prow["pick_id"]
                if pid in picks_by_id.index:
                    p = build_pick(picks_by_id.loc[pid], args.default_net)
                    if p is not None:
                        ev.picks.append(p)

        # Hypocenters
        hid = hypo_for_waveform.get(eid)
        if hid is not None:
            key = eid if hid in shared_hypo else None
            for _, orow in df_ho[df_ho["event_id"] == hid].iterrows():
                ev.origins.append(build_origin(orow, key))
            set_preferred_origin(ev, df_he[df_he['event_id'].eq(hid)].iloc[0], key)
            attach_preferred_magnitude(ev, df_he[df_he['event_id'].eq(hid)].iloc[0], key)
            ev.comments.append(Comment(text=f"hypocenter_link:{link_method.get(eid, association_policy)}"))
            if key is not None:
                others = [w for w in shared_hypo[hid] if w != eid]
                ev.comments.append(Comment(text=f"hypocenter_shared:{hid}; also_linked_to_events:{','.join(map(str, others))}"))

        # Final observed composition
        has_p = bool(ev.picks) or has_p_spine
        has_h = bool(ev.origins)

        if has_w and has_p and has_h:
            cls = "W_P_H"
        elif has_w and has_p:
            cls = "W_P"
        elif has_w and has_h:
            cls = "W_H"
        elif has_w:
            cls = "W_ONLY"
        elif has_p and has_h:
            cls = "P_H"
        elif has_p:
            cls = "P_ONLY"
        else:
            # A Step 32 event with neither waveform nor picks should not exist.
            continue

        ev.comments.append(Comment(text=f"event_class:{cls}"))
        ev.comments.extend(Comment(text=f'{key}:{value}') for key, value in
                           [('waveform_present', has_w), ('picks_present', has_p), ('hypocenter_present', has_h)])
        class_counts[cls] += 1
        catalog_all.events.append(ev)

    # ------------------------------------------------------------------
    # Hypocenter-only events
    # ------------------------------------------------------------------

    orphan_hypo_ids = set(df_he["event_id"]) - used_hypo_ids

    for hid in sorted(orphan_hypo_ids):
        ev = Event(resource_id=ResourceIdentifier(f"hypocenter/{hid}"))
        ev.comments = [Comment(text="event_class:H_ONLY"), Comment(text="waveform_present:False"),
                       Comment(text="picks_present:False"), Comment(text="hypocenter_present:True")]
        if hid in inside_record:
            # Origin falls inside a record already linked to another hypocentre (e.g. two events in one file).
            ev.comments.append(Comment(text=f"inside_record_of_event:{inside_record[hid]}"))

        for _, orow in df_ho[df_ho["event_id"] == hid].iterrows():
            ev.origins.append(build_origin(orow))

        set_preferred_origin(ev, df_he[df_he['event_id'].eq(hid)].iloc[0])

        attach_preferred_magnitude(ev, df_he[df_he['event_id'].eq(hid)].iloc[0])
        class_counts["H_ONLY"] += 1
        catalog_all.events.append(ev)

    # ------------------------------------------------------------------
    # catalog_waveform (authoritative waveform-related subset)
    # ------------------------------------------------------------------

    for ev in catalog_all:
        stabilize_comment_ids(ev)

    waveform_classes = {"W_P_H", "W_P", "W_H", "W_ONLY"}

    catalog_waveform = Catalog(
        events=[
            ev
            for ev in catalog_all
            if any(
                c.text == f"event_class:{cls}"
                for c in ev.comments
                for cls in waveform_classes
            )
        ]
    )

    # ------------------------------------------------------------------
    # Write outputs
    # ------------------------------------------------------------------

    out = Path(args.out_prefix)
    out.parent.mkdir(parents=True, exist_ok=True)

    all_xml = out.with_suffix(".xml")
    all_pkl = out.with_suffix(".pkl")
    wav_xml = out.with_name(out.name + "_waveform.xml")
    wav_pkl = out.with_name(out.name + "_waveform.pkl")

    export_classified_catalog(catalog_all, out.parent / (out.name + "_exports"), write_subsets=False)
    catalog_all.write(str(all_xml), format="QUAKEML")
    catalog_waveform.write(str(wav_xml), format="QUAKEML")

    with open(all_pkl, "wb") as f:
        pickle.dump(catalog_all, f)

    with open(wav_pkl, "wb") as f:
        pickle.dump(catalog_waveform, f)

    # ------------------------------------------------------------------
    # Reporting
    # ------------------------------------------------------------------

    print("\nSTEP 50 COMPLETE")
    print("----------------")
    print(f"Catalog ALL events:      {len(catalog_all)}")
    print(f"Catalog WAVEFORM events: {len(catalog_waveform)}")
    print("EVENT_CLASS breakdown (catalog_all):")
    for k in ["W_P_H", "W_P", "W_H", "W_ONLY", "P_H", "P_ONLY", "H_ONLY"]:
        print(f"  {k:7s}: {class_counts[k]}")

    if not matches.empty:
        print(f"{association_policy} spine↔hypocenter matches: {len(matches)}")
        print(f"Max accepted |origin - record start|: {matches['abs_dt_s'].max():.3f} s")
    else:
        print(f"{association_policy} spine↔hypocenter matches: 0")

    print("Written:")
    print(f"  {all_xml}")
    print(f"  {all_pkl}")
    print(f"  {wav_xml}")
    print(f"  {wav_pkl}")
    return {
        'catalog_all': catalog_all,
        'catalog_waveform': catalog_waveform,
        'matches': matches,
        'class_counts': class_counts,
        'hypocenters_inside_linked_records': inside_record,
        'association_policy': association_policy,
        'output_paths': {
            'all_xml': all_xml, 'all_pkl': all_pkl,
            'waveform_xml': wav_xml, 'waveform_pkl': wav_pkl,
        },
    }

## Optional command-line argument helpers

The next cell defines functions only. It does not execute the workflow or write outputs.

In [ ]:
def main(argv=None):
    """Compatibility CLI shared by the old script entry points."""
    import argparse
    parser = argparse.ArgumentParser(description='STEP 50: Build ObsPy catalogs')
    for name in ('waveform-event-index', 'waveform-pick-map', 'pick-index',
                 'hypo-event-index', 'hypo-origin-index', 'out-prefix'):
        parser.add_argument('--' + name, required=True)
    parser.add_argument('--origin-time-tol', type=float, default=10.0)
    parser.add_argument('--default-net', default='XB')
    parser.add_argument('--association-policy', choices=('picks_then_window', 'one_to_one', 'nearest'),
                        default='picks_then_window')
    parser.add_argument('--max-first-arrival-s', type=float, default=10.0)
    parser.add_argument('--origin-lead-s', type=float, default=10.0)
    return run(**vars(parser.parse_args(argv)))

## Run and inspect

Run this cell only after the configuration and all definitions. It writes the configured outputs; inspect the summary and exceptions before proceeding.

In [ ]:
result = run(**SETTINGS)

In [ ]:
display(result['association_policy'])
display(result['class_counts'])
display(result['matches'].head(20))
print('All events:', len(result['catalog_all']))
print('Waveform-related events:', len(result['catalog_waveform']))
display(result['output_paths'])

## Optional: QuakeML subsets and portable class manifest

The main run always writes the class manifest beside its catalogue outputs. Enable this cell to additionally write one QuakeML subset for each nonempty class and a `located` subset. Resource IDs and comments are preserved. These class subsets partition the all-record catalogue; `located` and `waveform` overlap them and must not be summed. This cell writes outputs but does not rerun assembly.

In [ ]:
RUN_ADDITIONAL_QUAKEML = False
EXPORT_DIR = Path(SETTINGS['out_prefix']).parent / (Path(SETTINGS['out_prefix']).name + '_exports')
if RUN_ADDITIONAL_QUAKEML:
    extra_exports = export_classified_catalog(result['catalog_all'], EXPORT_DIR, write_subsets=True)
    display(extra_exports['quakeml_paths'])
else:
    print('Optional QuakeML subsets disabled; main-run all/waveform catalogues remain available.')